# IMPORTS

In [ ]:
import pandas as pd
import numpy as np

# Loading the DataFrame

In [2]:
df = pd.read_csv('../data/final/master_football_dataset.csv')

# Adding Features to Dataset

In [3]:
# EFFICIENCY METRICS

# Points per €100M of squad value
df['points_per_100m_squad'] = (
    df['points'] / (df['squad_market_value'] / 1e8)
).round(2)

# Points per €10M of transfer spending (only where spending > 0)
df['points_per_10m_spend'] = np.where(
    df['transfer_spending'] >= 10_000_000,
    df['points'] / (df['transfer_spending'] / 1e7),
    np.nan
)

# RELATIVE RANK OF EACH CLUB WITHIN ITS LEAGUE-SEASON

# Spending rank (1 = highest spender in that league that season)
df['spending_rank'] = (
    df.groupby(['competition_id', 'year'])['transfer_spending']
    .rank(ascending=False, method='min')
    .astype(int)
)

# Squad value rank (1 = richest squad)
df['value_rank'] = (
    df.groupby(['competition_id', 'year'])['squad_market_value']
    .rank(ascending=False, method='min')
    .astype(int)
)

# Points rank (1 = champion)
df['points_rank'] = (
    df.groupby(['competition_id', 'year'])['points']
    .rank(ascending=False, method='min')
    .astype(int)
)

# RANK DIFFERENCES (OVER AND UNDERPERFORMANCE)

# Positive = overperforming (ranked higher in points than in squad value)
# Negative = underperforming
df['value_vs_points_rank'] = df['value_rank'] - df['points_rank']
df['spend_vs_points_rank']  = df['spending_rank'] - df['points_rank']

# BINARY FLAGS

# Title won
df['title_won'] = (df['position'] == 1).astype(int)

# Top 4 finish (Champions League qualification in most leagues)
df['top4_finish'] = (df['position'] <= 4).astype(int)

# Relegated — bottom 3 for 20-team leagues, bottom 2 for 18-team leagues
df['relegated'] = 0
df.loc[(df['competition_id'] == 'L1') & (df['position'] >= 17), 'relegated'] = 1
df.loc[(df['competition_id'] != 'L1') & (df['position'] >= 18), 'relegated'] = 1

# Middle East ownership (1 = owned by Middle Eastern investors, 0 = not)

df['middle_east_owned'] = 0
df.loc[df['club_id'].isin([281, 583]), 'middle_east_owned'] = 1
df.loc[(df['club_id'] == 762) & (df['year'] >= 2021), 'middle_east_owned'] = 1

# FINANCIAL INEQUALITY INDICATORS 

# Each club's squad value relative to league average that season
league_avg = df.groupby(['competition_id','year'])['squad_market_value'].transform('mean')
df['value_vs_league_avg'] = ((df['squad_market_value'] - league_avg) / league_avg * 100).round(1)

# Each club's spending relative to league average that season  
spend_avg = df.groupby(['competition_id','year'])['transfer_spending'].transform('mean')
df['spend_vs_league_avg'] = ((df['transfer_spending'] - spend_avg) / spend_avg * 100).round(1)

# YEAR-ON-YEAR CHANGES

df = df.sort_values(['club_id', 'year'])

# Change in squad value vs previous season
df['squad_value_change'] = df.groupby('club_id')['squad_market_value'].diff()
df['squad_value_change_pct'] = df.groupby('club_id')['squad_market_value'].pct_change() * 100

# Change in points vs previous season
df['points_change'] = df.groupby('club_id')['points'].diff()

# after year-on-year changes
df.to_csv('../data/final/master_football_engineered.csv', index=False)
print(f"✓ Saved: {df.shape}")
print(f"Columns: {df.columns.tolist()}")

✓ Saved: (978, 36)
Columns: ['year', 'competition_id', 'club_id', 'club_name', 'matches_played', 'wins', 'draws', 'losses', 'goals_for', 'goals_against', 'points', 'goal_difference', 'position', 'league', 'country_name', 'transfer_spending', 'squad_market_value', 'num_players_valued', 'season', 'clean_sheets', 'points_per_100m_squad', 'points_per_10m_spend', 'spending_rank', 'value_rank', 'points_rank', 'value_vs_points_rank', 'spend_vs_points_rank', 'title_won', 'top4_finish', 'relegated', 'middle_east_owned', 'value_vs_league_avg', 'spend_vs_league_avg', 'squad_value_change', 'squad_value_change_pct', 'points_change']


SABINA COMMENT: Damn, both you and Reynold really went for it in defining your own metrics, I love it!

Nothing to add, looking good!